In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(
    pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1),
    future=True,
)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 80)


def q(sql: str) -> pd.DataFrame:
    """DQL-запит → DataFrame."""
    return pd.read_sql(text(sql), engine)


def run(sql: str) -> None:
    """DDL / DML без результату, в одній транзакції."""
    with engine.begin() as conn:
        conn.execute(text(sql))


def dml(sql: str) -> pd.DataFrame:
    """DML з RETURNING: виконати в транзакції й повернути повернуті рядки як DataFrame."""
    with engine.begin() as conn:
        result = conn.execute(text(sql))
        if result.returns_rows:
            return pd.DataFrame(result.mappings().all(), columns=list(result.keys()))
        return pd.DataFrame({'rowcount': [result.rowcount]})


def show_returning(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """Audit-log: кількість повернутих рядків + перші n."""
    print(f'RETURNING: {len(df)} row(s)')
    return df.head(n)

Note: you may need to restart the kernel to use updated packages.


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
FILES = {
    'olist_customers_raw':     'olist_customers_dataset.csv',
    'olist_orders_raw':        'olist_orders_dataset.csv',
    'olist_order_items_raw':   'olist_order_items_dataset.csv',
    'olist_products_raw':      'olist_products_dataset.csv',
    'olist_sellers_raw':       'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

REPO_URL = 'https://github.com/<ВАШ_ЛОГІН>/goit-rdb-hw-04.git'   # замініть логін
CANDIDATE_DIRS = ['data/olist', '/content/goit-rdb-hw-04/data/olist']


def has_all_files(d):
    return all(os.path.exists(os.path.join(d, f)) for f in FILES.values())


dataset_dir = next((d for d in CANDIDATE_DIRS if has_all_files(d)), None)

if dataset_dir is None and '<' not in REPO_URL:
    import subprocess
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO_URL, '/content/goit-rdb-hw-04'], check=False)
    dataset_dir = next((d for d in CANDIDATE_DIRS if has_all_files(d)), None)

if dataset_dir is not None:
    print('Варіант B: CSV з репозиторію')
else:
    import kagglehub
    dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    print('Варіант A: kagglehub (версія з Kaggle може відрізнятися від знімка в репозиторії)')

print(dataset_dir)
print(sorted(f for f in os.listdir(dataset_dir) if f.endswith('.csv')))

Варіант B: CSV з репозиторію
data/olist
['olist_customers_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv']


In [3]:
raw_shapes = {}
for table_name, file_name in FILES.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    raw_shapes[table_name] = df.shape
    print(f'{table_name:<25} {df.shape}')

olist_customers_raw       (99441, 5)


olist_orders_raw          (99441, 8)


olist_order_items_raw     (112650, 7)


olist_products_raw        (32951, 9)
olist_sellers_raw         (3095, 4)


olist_order_reviews_raw   (100000, 7)


In [4]:
run(r"""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
""")

In [5]:
run(r"""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")

In [6]:
q(r"""
SELECT t.table_name, t.raw_rows, t.typed_rows, t.raw_rows - t.typed_rows AS lost_rows
FROM (
    SELECT 'customers'   AS table_name, (SELECT COUNT(*) FROM olist_customers_raw)     AS raw_rows, (SELECT COUNT(*) FROM olist_customers)     AS typed_rows
    UNION ALL SELECT 'orders',      (SELECT COUNT(*) FROM olist_orders_raw),        (SELECT COUNT(*) FROM olist_orders)
    UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw),   (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL SELECT 'products',    (SELECT COUNT(*) FROM olist_products_raw),      (SELECT COUNT(*) FROM olist_products)
    UNION ALL SELECT 'sellers',     (SELECT COUNT(*) FROM olist_sellers_raw),       (SELECT COUNT(*) FROM olist_sellers)
    UNION ALL SELECT 'reviews',     (SELECT COUNT(*) FROM olist_order_reviews_raw), (SELECT COUNT(*) FROM olist_order_reviews)
) AS t
ORDER BY t.table_name;
""")

,table_name,raw_rows,typed_rows,lost_rows
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,reviews,100000,100000,0
5,sellers,3095,3095,0


**Висновок:** у всіх шести таблицях кількість рядків у typed-шарі **дорівнює** raw: 99 441 customers і orders, 112 650 order_items, 32 951 products, 3 095 sellers, 100 000 reviews. Жоден рядок не втрачено: усі приведення типів пройшли, усі FK знайшли батьківські записи, `CHECK` (статуси, невід'ємні ціни, `review_score` 1–5) не відхилили жодного рядка. Це очікувано, бо `INSERT ... SELECT` не має фільтрів `WHERE`: якби якийсь рядок порушив constraint, впала б уся транзакція, а не зник окремий рядок.

In [7]:
run(r"""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [8]:
res = dml(r"""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
""")
show_returning(res, 20)

RETURNING: 1 row(s)


,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-10-09 23:12:14.970267+03:00


**Що змінює:** вставляє один рядок у `seller_alerts`: продавець `4a3ca9315b744ce9f8e9374361493884` має **172** замовлення, доставлені пізніше обіцяної дати (порівняння за `::DATE`, тобто запізнення хоча б на день).

**Навіщо `RETURNING`:** база одразу повертає згенеровані нею значення (`alert_id` з identity, `created_at` з `DEFAULT NOW()`) без окремого `SELECT`. Цей результат можна одразу показати в notebook як audit-log або передати далі в pipeline (наприклад, надіслати сповіщення з конкретним `alert_id`).

**Повторний запуск:** цей `INSERT` **не** ідемпотентний: кожен запуск додає новий alert. Тут це прийнятно, бо таблиця перестворюється в попередній клітинці; у production варто додати `UNIQUE (seller_id, alert_type)` + `ON CONFLICT`.

In [9]:
run(r"""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

In [10]:
SELLER_SCORE_SQL = r"""
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
"""

res_first = dml(SELLER_SCORE_SQL)
show_returning(res_first, 10)

RETURNING: 3095 row(s)


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-09 23:12:15.075221+03:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-09 23:12:15.075221+03:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-09 23:12:15.075221+03:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-09 23:12:15.075221+03:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-09 23:12:15.075221+03:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-09 23:12:15.075221+03:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-09 23:12:15.075221+03:00
7,b39d7fe263ef469605dbb32608aee0af,3.86,44,silver,2026-10-09 23:12:15.075221+03:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-09 23:12:15.075221+03:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-09 23:12:15.075221+03:00


In [11]:
res_second = dml(SELLER_SCORE_SQL)
print('RETURNING on 2nd run:', len(res_second))
q('''
SELECT
    COUNT(*)                                   AS n_rows,
    COUNT(DISTINCT seller_id)                  AS n_distinct_sellers,
    COUNT(*) FILTER (WHERE tier IS NULL)       AS n_without_tier,
    COUNT(*) FILTER (WHERE n_reviews = 0)      AS n_zero_reviews
FROM seller_score
''')

RETURNING on 2nd run: 3095


,n_rows,n_distinct_sellers,n_without_tier,n_zero_reviews
0,3095,3095,0,0


In [12]:
q(r"""
SELECT
    COALESCE(tier, '(no reviews)')      AS tier,
    COUNT(*)                            AS n_sellers,
    ROUND(AVG(avg_rating), 2)           AS mean_avg_rating,
    SUM(n_reviews)                      AS total_reviewed_orders
FROM seller_score
GROUP BY tier
ORDER BY n_sellers DESC;
""")

,tier,n_sellers,mean_avg_rating,total_reviewed_orders
0,silver,1572,4.06,88073
1,gold,968,4.83,6758
2,bronze,555,2.30,5179


**Результат:** у `seller_score` **3 095** рядків, по одному на кожного продавця, який має хоча б одну позицію. Другий запуск повернув ті самі 3 095 рядків через `RETURNING`, але кількість рядків у таблиці не змінилася: це `UPDATE` наявних, а не нові вставки. Розподіл: silver - 1 572, gold - 968, bronze - 555. Gold-продавці мають у сумі лише 6 758 оцінених замовлень (≈ 7 на продавця), а silver - 88 073. Отже, високий рейтинг часто є ефектом малої вибірки, і для production варто додати мінімальний поріг `n_reviews` для gold.

- **Чому `SELECT DISTINCT` у `seller_orders`:** `order_items` має грануляність «позиція замовлення». Якщо продавець продав 3 товари в одному замовленні, без `DISTINCT` замовлення (і його відгук) рахувалося б 3 рази. `DISTINCT` зводить дані до `seller_id × order_id`.
- **Чому review спочатку агрегується на рівні `order_id`:** у джерелі одне замовлення може мати кілька review-записів (555 таких замовлень, див. 6.4). `review_per_order` дає рівно одну оцінку на замовлення, тож при join з `seller_orders` грануляність не множиться.
- **Чому `ON CONFLICT (seller_id) DO UPDATE` ідемпотентний:** `seller_id` - PK. При повторному запуску кожен рядок конфліктує з наявним і замість дубліката оновлює метрики з `EXCLUDED`. Скільки б разів не запускати, у таблиці буде рівно один рядок на продавця з актуальними даними.
- **Чому можливі `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`:** `LEFT JOIN` залишає продавців, чиї замовлення не мають оцінки. `AVG` по порожній множині дає `NULL`, `COUNT(rpo.order_id)` дає 0, а `CASE` повертає `NULL` для tier. Чесніше показати «немає даних», ніж приписати 0 (найгіршу оцінку). **У нашому знімку таких продавців 0**, бо кожне замовлення має відгук (див. 4.5).
- **Обмеження датасету:** review прив'язаний до замовлення, а не до продавця. У 1 278 замовленнях кілька продавців, і їхній відгук зараховано кожному з них. Тому сума `total_reviewed_orders` (100 010) більша за кількість замовлень з позиціями (98 666). Це навчальне спрощення: оцінку за погану доставку одного продавця отримує й інший.

In [13]:
run(r"""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
""")

In [14]:
res = dml(r"""
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
""")
show_returning(res, 5)

RETURNING: 99441 row(s)


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False


In [15]:
q(r"""
SELECT
    is_late,
    COUNT(*)                                                         AS n_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)               AS pct_orders,
    COUNT(*) FILTER (WHERE order_status = 'delivered')               AS n_delivered
FROM olist_orders
GROUP BY is_late
ORDER BY is_late;
""")

,is_late,n_orders,pct_orders,n_delivered
0,False,92906,93.43,89944
1,True,6535,6.57,6534


**Що змінює:** для всіх 99 441 замовлень `is_late` перераховується з дат доставки (`RETURNING` повернув 99 441 рядок, показано перші 5). Запізнилися **6 535** замовлень (6,57 %). З них 6 534 мають статус `delivered`, а одне - інший статус, хоча має дату доставки (невідповідність статусу й дат, див. 6.4).

**Чому безпечно перезапускати:** значення не «дописується», а **обчислюється заново** з поточних дат для кожного рядка. Повторний запуск дає той самий результат, а якщо дата доставки зміниться, прапорець коректно перейде і з `FALSE` у `TRUE`, і назад. `ADD COLUMN IF NOT EXISTS` теж не падає при повторі.

In [16]:
res = dml(r"""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")
show_returning(res, 20)

RETURNING: 0 row(s)


,alert_id,seller_id,alert_type,created_at


In [17]:
q(r"""
SELECT COUNT(*) AS alerts_left FROM seller_alerts;
""")

,alerts_left
0,1


**Результат порожній - це очікувано.** Єдиний alert створено кілька секунд тому в 2.1, тож умова `created_at < NOW() - INTERVAL '30 days'` не виконується, і в таблиці лишився 1 рядок. `RETURNING` тут працює як audit-log видалення: якби рядки видалялися, ми б одразу побачили, які саме `alert_id` зникли, і могли б зберегти їх в архів. Запит ідемпотентний, бо повторний запуск видалить лише те, що «постаріло» на момент запуску.

In [18]:
run(r"""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")

In [19]:
q(r"""
SELECT
    (SELECT COUNT(*) FROM olist_customers)                       AS n_customer_id,
    (SELECT COUNT(*) FROM olist_customer_dim)                    AS n_customer_unique_id,
    (SELECT COUNT(*) FROM (
        SELECT customer_unique_id
        FROM olist_customers
        GROUP BY customer_unique_id
        HAVING COUNT(*) >= 2
    ) AS t)                                                      AS n_unique_with_2plus_ids;
""")

,n_customer_id,n_customer_unique_id,n_unique_with_2plus_ids
0,99441,96096,2997


**Висновок:** 99 441 `customer_id` відповідають лише **96 096** реальним покупцям (`customer_unique_id`), а 2 997 покупців мають два чи більше `customer_id`. Olist генерує новий `customer_id` для кожного замовлення, тому без дедуплікації повторні покупці виглядали б як різні люди. `DISTINCT ON` залишає для кожного покупця рядок з **найновішим** замовленням (`ORDER BY ... order_purchase_timestamp DESC NULLS LAST`), тож штат і місто в dimension - найактуальніші.

In [20]:
run(r"""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")

In [21]:
SEG_SQL = r"""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
"""

res_seg = dml(SEG_SQL)
show_returning(res_seg, 10)

RETURNING: 96096 row(s)


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


In [22]:
max_id_before = q('SELECT MAX(segment_id) AS m FROM customer_segments').m[0]
res_seg_2 = dml(SEG_SQL)
q(f'''
SELECT
    COUNT(*)                         AS n_rows,
    COUNT(DISTINCT customer_unique_id) AS n_distinct_customers,
    MAX(segment_id)                  AS max_segment_id,
    {max_id_before}                  AS max_segment_id_before_rerun
FROM customer_segments
''')

,n_rows,n_distinct_customers,max_segment_id,max_segment_id_before_rerun
0,96096,96096,96096,96096


In [23]:
q(r"""
SELECT
    segment_name,
    rfm_score,
    COUNT(*)                                             AS n_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)   AS pct_customers,
    ROUND(AVG(monetary_value), 2)                        AS avg_monetary,
    ROUND(AVG(n_orders), 2)                              AS avg_orders,
    ROUND(AVG(recency_days), 0)                          AS avg_recency_days
FROM customer_segments
GROUP BY segment_name, rfm_score
ORDER BY rfm_score DESC, segment_name;
""")

,segment_name,rfm_score,n_customers,pct_customers,avg_monetary,avg_orders,avg_recency_days
0,VIP,5,1191,1.24,1591.81,1.11,290.0
1,new,4,269,0.28,675.09,1.00,74.0
2,regular,4,2964,3.08,680.51,1.14,305.0
3,regular,3,2608,2.71,220.74,2.07,271.0
4,inactive,2,40032,41.66,59.28,1.00,316.0
5,new,2,8622,8.97,123.71,1.00,72.0
6,regular,2,40410,42.05,191.38,1.00,310.0


In [24]:
q(r"""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'new'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_uniq...


**Результат сегментації:** 96 096 покупців. Найбільші сегменти - `regular` з rfm 2 (42,05 %) та `inactive` (41,66 %). `new` - 9,25 % (8 622 + 269), VIP - 1,24 % (1 191 покупець, середній монетарний обсяг 1 592 BRL). Повторний запуск не змінив ні кількість рядків, ні `MAX(segment_id)` = 96 096: `ON CONFLICT` оновив наявні рядки, і жоден identity-номер не витрачено на нові вставки. `rfm_score = 1` не трапляється, бо кожен покупець у dimension має хоча б одне замовлення. Слабке місце правил: 40 410 «regular» мають лише одне замовлення і потрапили в сегмент тільки через `monetary ≥ 100`.

Constraints у `pg_constraint`: `p` - PK `segment_id`, `u` - `UNIQUE (customer_unique_id)`, `f` - FK на `olist_customer_dim` з `ON DELETE CASCADE`, `c` - п'ять CHECK (segment_name, rfm_score, recency_days, monetary_value, n_orders). `NOT NULL` і `DEFAULT` зберігаються в `pg_attribute`, а не в `pg_constraint`, тому тут їх не видно.

- **Чому `customer_unique_id`:** `customer_id` створюється заново для кожного замовлення. Сегментація за ним дала б n_orders = 1 майже для всіх, і RFM-frequency втратила б сенс. `customer_unique_id` - це реальна людина.
- **Чому потрібна `olist_customer_dim` для FK:** FK може посилатися лише на `PRIMARY KEY`/`UNIQUE`-колонку. В `olist_customers` `customer_unique_id` не унікальний (2 997 повторів), тож посилатися на нього неможливо. Dimension, де `customer_unique_id` є PK, дає цільову таблицю для FK і гарантує, що сегмент існує лише для відомого покупця.
- **Чому `reference_date` з датасету:** дані історичні (останнє замовлення - 2018 р.). `CURRENT_DATE` дав би recency 2 000+ днів для всіх, усі покупці стали б «давніми», а результат змінювався б щодня без зміни даних. `MAX(order_purchase_timestamp) + 1` робить розрахунок детермінованим і відтворюваним.
- **Як `ON CONFLICT` робить сегментацію повторюваною:** `UNIQUE (customer_unique_id)` - ціль конфлікту. Новий покупець вставляється, а наявний оновлюється (`segment_name`, `rfm_score`, метрики, `assigned_at`). Тож клітинку можна перезапускати після кожного оновлення даних без `TRUNCATE` і без дублікатів.

In [25]:
q(r"""
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** усі 10 найприбутковіших комбінацій - це продавці зі **штату SP**. Лідери - `cama_mesa_banho` (постільна білизна, ~1,07 млн BRL, 8 214 замовлень) і `relogios_presentes` (годинники й подарунки, ~1,02 млн BRL за майже вдвічі меншої кількості замовлень, тобто дорожчий середній чек). SP домінує як центр постачання, тому для порівняння категорій без регіонального зсуву варто нормалізувати метрики за штатом продавця.

In [26]:
q(r"""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,1.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Interpretation:** найбільший сумарний чек - **13 664 BRL** (одне замовлення, RJ). Серед топ-20 переважають покупці з одним замовленням, лише кілька мають 2–4. Помітно, що багато найдорожчих покупців поставили оцінку **1.0**: дорогі замовлення частіше розчаровують, і це сигнал для churn-моделі. `review_per_order` спершу зводить відгуки до одного значення на замовлення, тому замовлення з кількома позиціями чи кількома review не «важить» більше. Застереження щодо `COALESCE(avg_review_score, 0)`: нуль змішує «немає відгуку» з «дуже погано». Для ML краще залишити `NULL` або додати прапорець `has_review`.

In [27]:
q(r"""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
""")

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Interpretation:** з 27 штатів **23** мають і клієнтів, і продавців, а **4** (AL, AP, RR, TO) - лише клієнтів. Штатів «лише продавці» немає. Концентрація дуже висока: у SP 41 746 клієнтів і 1 849 продавців (≈ 60 % усіх продавців), а в 5 штатах (AC, AM, MA, PA, PI) є лише по одному продавцю. Для північних і північно-східних штатів майже всі замовлення йдуть міжрегіональною доставкою, що пояснює вищий freight і AOV (див. 5.2).

In [28]:
q(r"""
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


In [29]:
q(r"""
SELECT
    s1.seller_state,
    COUNT(*) AS n_pairs_different_city
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
GROUP BY s1.seller_state
ORDER BY n_pairs_different_city DESC
LIMIT 5;
""")

,seller_state,n_pairs_different_city
0,SP,1457751
1,PR,51665
2,MG,27184
3,SC,17208
4,RJ,10151


**Interpretation:** self-join генерує пари продавців одного штату з різних міст. Перші рядки - Баїя (BA), наприклад `lauro de freitas` ↔ `salvador`. Масштаб величезний: лише в SP **1 457 751** таких пар, у PR - 51 665. Тому self-join без `LIMIT` чи додаткових фільтрів (одна категорія, схожий обсяг продажів) непридатний для практичного порівняння. У результаті видно й проблему якості: місто `bahia` - це назва штату, записана як місто.

In [30]:
q(r"""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
""")

,order_id,customer_id,order_status,order_purchase_timestamp


In [31]:
q(r"""
SELECT
    o.order_status,
    COUNT(*) AS n_orders_without_review
FROM olist_orders AS o
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_order_reviews AS rv
    WHERE rv.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY n_orders_without_review DESC;
""")

,order_status,n_orders_without_review


**Interpretation:** обидва anti-join (`LEFT JOIN ... IS NULL` та `NOT EXISTS`) повернули **порожній результат**: у використаному знімку датасету (100 000 review) **кожне** з 99 441 замовлень має хоча б один review-запис, навіть скасовані й недоставлені. Отже, відгук тут не є сигналом «покупець залишив думку» - Olist генерує анкету для кожного замовлення, а пропуском радше є порожній текст коментаря (58 % записів, див. 6.4). Порожній anti-join теж корисний: він підтверджує, що `review_count = 0` як ML-фіча в цьому знімку завжди дорівнюватиме нулю й нічого не дасть.

In [32]:
q(r"""
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** поріг 10 000 BRL проходять **53** категорії, включно з товарами без категорії (`NULL`, ~180 тис. BRL - потрібна окрема мітка). Лідер за gross revenue - `beleza_saude` (1,26 млн), за ним `relogios_presentes` (1,21 млн) і `cama_mesa_banho` (1,04 млн). Останні мають найбільше замовлень (9 417), але низьку середню ціну (93 BRL) і високу частку доставки: freight ≈ 20 % від ціни. Найвищий середній чек позиції - у `pcs` (1 098 BRL) за лише 181 замовлення.

In [33]:
q(r"""
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Interpretation:** поріг ≥ 100 замовлень проходять **24** штати. Найвищий AOV - у **PB (265 BRL)**, AL (234) і RO (233), а найнижчий - у **SP (143 BRL)**, хоча SP дає найбільший GMV (5,9 млн, 41 375 замовлень). Віддалені від SP штати мають вищий середній чек, бо до нього входить дорожча міжрегіональна доставка, а покупці там, імовірно, частіше замовляють дорогі товари, яких немає локально. `HAVING` відкидає штати з малою кількістю замовлень, де AOV нестабільний через кілька великих покупок.

In [34]:
q(r"""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
""")

,customer_unique_id,n_orders,total_spend,categories
0,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
1,c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
2,055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
3,59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
4,58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
5,d77aa95864ae5b42160937615628723a,2,2400.00,pcs
6,7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
7,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
8,6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
9,73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


In [35]:
q(r"""
SELECT
    COUNT(*)                                                       AS repeat_customers,
    COUNT(*) FILTER (WHERE n_categories = 1)                       AS single_category,
    ROUND(COUNT(*) FILTER (WHERE n_categories = 1) * 100.0 / COUNT(*), 2) AS pct_single_category
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT COALESCE(p.product_category_name, '(unknown)')) AS n_categories
    FROM olist_customers AS c
    JOIN olist_orders AS o      ON o.customer_id = c.customer_id
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY c.customer_unique_id
    HAVING COUNT(DISTINCT o.order_id) >= 2
) AS t;
""")

,repeat_customers,single_category,pct_single_category
0,2913,1275,43.77


**Interpretation:** повторних покупців з позиціями - **2 913** (≈ 3 % від 96 096), тобто Olist майже повністю тримається на разових покупках. Найцінніший повторний покупець витратив 7 388 BRL на `eletroportateis`. **43,77 %** повторних покупців повертаються в **ту саму** категорію, а решта купують у 2+ категоріях (наприклад, `moveis_sala, utilidades_domesticas`). Отже, для рекомендацій варто комбінувати «ще з тієї ж категорії» з cross-category підбором.

In [36]:
q(r"""
SELECT
    'HAVING only (all statuses)' AS variant,
    COUNT(*)                     AS n_categories_passed,
    SUM(gross_revenue)           AS total_revenue
FROM (
    SELECT p.product_category_name, SUM(oi.price) AS gross_revenue
    FROM olist_order_items AS oi
    JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY p.product_category_name
    HAVING SUM(oi.price) > 10000
) AS a

UNION ALL

SELECT
    'WHERE delivered + HAVING',
    COUNT(*),
    SUM(gross_revenue)
FROM (
    SELECT p.product_category_name, SUM(oi.price) AS gross_revenue
    FROM olist_order_items AS oi
    JOIN olist_orders   AS o ON o.order_id = oi.order_id
    JOIN olist_products AS p ON p.product_id = oi.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
    HAVING SUM(oi.price) > 10000
) AS b;
""")

,variant,n_categories_passed,total_revenue
0,HAVING only (all statuses),53,13513984.94
1,WHERE delivered + HAVING,53,13146723.69


**WHERE vs HAVING:** `WHERE o.order_status = 'delivered'` фільтрує **рядки до агрегації**: позиції скасованих, недоставлених тощо замовлень взагалі не потрапляють у `SUM`, і загальний revenue категорій падає з 13,51 до 13,15 млн BRL (−367 тис.). `HAVING SUM(oi.price) > 10000` фільтрує **групи після агрегації**: він вирішує, чи показувати категорію, дивлячись на її підсумок. У нашому випадку обидва варіанти пропускають 53 категорії, бо недоставлені замовлення не опускають жодну категорію нижче порогу, але суми в них уже інші. `HAVING` не можна замінити на `WHERE`: у `WHERE` ще немає `SUM`.

In [37]:
q(r"""
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
""")

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


**Interpretation:** `UNION` повернув **50** унікальних пар (штат, роль): 27 рядків `customer` + 23 `seller`. Чотири штати (AL, AP, RR, TO) мають лише рядок `customer`. Результат той самий, що й у `FULL OUTER JOIN` (4.3), але у «довгому» форматі без лічильників.

In [38]:
q(r"""
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;
""")

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


In [39]:
q(r"""
SELECT customer_state AS state_without_sellers
FROM olist_customers

EXCEPT

SELECT seller_state
FROM olist_sellers
ORDER BY state_without_sellers;
""")

,state_without_sellers
0,AL
1,AP
2,RR
3,TO


**Interpretation:** `INTERSECT` дає **23** штати з обома ролями, `EXCEPT` - **4** штати без жодного продавця (AL, AP, RR, TO). Це рівно ті самі множини, що й `both` та `customers_only` у FULL OUTER JOIN, тобто дві техніки дають узгоджений результат. Set-оператори коротші й одразу усувають дублікати, а FULL OUTER JOIN дозволяє зберегти кількості з обох боків.

In [40]:
q(r"""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


In [41]:
q(r"""
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    COUNT(*)                         AS n_orders,
    COUNT(*) FILTER (WHERE rn = 1)   AS n_latest_orders,
    COUNT(*) FILTER (WHERE rn >= 2)  AS n_older_orders,
    MAX(rn)                          AS max_orders_per_customer
FROM ranked_orders;
""")

,n_orders,n_latest_orders,n_older_orders,max_orders_per_customer
0,99441,96096,3345,17


**Interpretation:** з 99 441 замовлень `ROW_NUMBER() = 1` отримали **96 096**, рівно по одному на кожного `customer_unique_id`. Ще 3 345 - це попередні замовлення повторних покупців. Рекордсмен має **17** замовлень. Сортування з `order_id DESC` як tie-breaker робить вибір детермінованим, навіть якщо два замовлення мають однаковий timestamp. Саме так у pipeline беруть «останній стан» покупця для recency-фічі чи для `latest_customer_id` у dimension.

In [42]:
q(r"""
SELECT 'products without category'            AS check_name,
       COUNT(*) FILTER (WHERE product_category_name IS NULL) AS n_bad,
       COUNT(*) AS n_total
FROM olist_products
UNION ALL
SELECT 'orders without any review',
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM olist_order_reviews rv WHERE rv.order_id = o.order_id)),
       COUNT(*)
FROM olist_orders AS o
UNION ALL
SELECT 'orders with 2+ reviews',
       COUNT(*) FILTER (WHERE n > 1), COUNT(*)
FROM (SELECT order_id, COUNT(*) AS n FROM olist_order_reviews GROUP BY order_id) AS r
UNION ALL
SELECT 'review_id used more than once',
       COUNT(*) FILTER (WHERE n > 1), COUNT(*)
FROM (SELECT review_id, COUNT(*) AS n FROM olist_order_reviews GROUP BY review_id) AS r
UNION ALL
SELECT 'reviews with empty comment message',
       COUNT(*) FILTER (WHERE NULLIF(TRIM(review_comment_message), '') IS NULL), COUNT(*)
FROM olist_order_reviews
UNION ALL
SELECT 'orders without items',
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM olist_order_items oi WHERE oi.order_id = o.order_id)),
       COUNT(*)
FROM olist_orders AS o
UNION ALL
SELECT 'delivered orders without delivery date',
       COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL),
       COUNT(*) FILTER (WHERE order_status = 'delivered')
FROM olist_orders
UNION ALL
SELECT 'non-delivered orders WITH delivery date',
       COUNT(*) FILTER (WHERE order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL),
       COUNT(*) FILTER (WHERE order_status <> 'delivered')
FROM olist_orders
UNION ALL
SELECT 'orders with 2+ sellers',
       COUNT(*) FILTER (WHERE n_sellers > 1), COUNT(*)
FROM (SELECT order_id, COUNT(DISTINCT seller_id) AS n_sellers FROM olist_order_items GROUP BY order_id) AS m;
""")

,check_name,n_bad,n_total
0,products without category,610,32951
1,orders without any review,0,99441
2,orders with 2+ reviews,555,99441
3,review_id used more than once,802,99173
4,reviews with empty comment message,58256,100000
5,orders without items,775,99441
6,delivered orders without delivery date,8,96478
7,non-delivered orders WITH delivery date,6,2963
8,orders with 2+ sellers,1278,98666


In [43]:
q(r"""
SELECT order_status, COUNT(*) AS n_orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC;
""")

,order_status,n_orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


**Висновок:** відсутніх review немає, але є **555** замовлень з 2+ відгуками, а **802** `review_id` використано повторно (тому PK - технічний `review_row_id`). **58 %** відгуків без тексту коментаря. **610** товарів (1,85 %) без категорії. **775** замовлень зовсім без позицій (603 `unavailable`, 164 `canceled`), і в RFM вони мають `monetary = 0`. Статуси й дати не завжди узгоджені: 8 `delivered` без дати доставки та 6 не-`delivered` з датою доставки. 97 % замовлень мають статус `delivered`, тож класи статусів сильно незбалансовані. **1 278** замовлень мають кількох продавців: саме вони створюють неоднозначність при прив'язці відгуку до продавця.

**Найскладніший JOIN** - `LEFT JOIN` для seller- і customer-фіч. Синтаксично він простий, але легко помилитися з грануляністю: join `order_items → reviews` тихо множить оцінки, і результат виглядає правдоподібно. Тому треба постійно ставити питання «що є одним рядком?». Найважче було сформулювати business-question для **self-join**: технічно пари продавців генеруються легко, але без додаткових фільтрів (1,46 млн пар лише в SP) питання «кого з ким порівнювати» не має практичної відповіді.

**Найкорисніша фіча для production-ETL** - `ON CONFLICT`. Разом з `UNIQUE`/PK він робить завантаження ідемпотентним: перезапуск після збою не створює дублікатів. `RETURNING` доповнює його готовим audit-log без зайвого `SELECT`.

**Window functions у ML-проєкті:** останнє замовлення покупця (`ROW_NUMBER`), номер покупки та час між покупками (`LAG`), ковзні середні оцінок продавця, ранжування продавців у категорії (`RANK`), а також point-in-time фічі, де вікно обмежене датою прогнозу, щоб уникнути leakage.

**Data-quality:** 555 замовлень з кількома відгуками й 802 повторних `review_id`, 58 % порожніх коментарів, 610 товарів без категорії, 775 замовлень без позицій, суперечності між статусом і датами доставки, помилка `lenght` у назвах колонок, назва штату в полі міста.

**`customer_unique_id`**, бо `customer_id` новий для кожного замовлення: frequency вироджується до 1, і повторні покупці (2 997) губляться.

**Грануляність `seller_id × order_id`** потрібна тому, що позицій у замовленні може бути кілька. Без `DISTINCT` і попередньої агрегації відгуків одна оцінка рахувалася б стільки разів, скільки товарів продавець поклав у замовлення, і рейтинг зміщувався б на користь великих кошиків.